<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/SQL_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import re
import operator

class SQLParser:
    def _convert(self, v):
        if v.isdigit:
            return int(v)
        try:
            return float(v)
        except:
            return v

    def _compile_where(self, where):
        ops = {
            "=": operator.__eq__,
            "!=": operator.__ne__,
            ">": operator.__gt__,
            ">=": operator.__ge__,
            "<": operator.__lt__,
            "<=": operator.__le__,
        }
        return (where[0], lambda a: ops[where[1]](a, self._convert(where[2])))

    def __init__(self, query):
        self.select = []
        self.where = []
        self.order_by = None

        query_tokens = re.split(r"(SELECT|WHERE|ORDER BY)", query, flags=re.I)
        for i in range(1, len(query_tokens)-1):
            if query_tokens[i] == "SELECT":
                selection = query_tokens[i+1].replace(" ", "")
                if selection != "*":
                    self.select = operator.itemgetter(*selection.split(","))
            if query_tokens[i] == "WHERE":
                whereclauses = re.split(r"AND", query_tokens[i+1])
                whereclauses = [w.replace(" ", "") for w in whereclauses]
                whereclauses = [re.split(r"([<>!=]{1,2})", w) for w in whereclauses]
                self.where = [self._compile_where(w) for w in whereclauses]
            if query_tokens[i] == "ORDER BY":
                order_by = query_tokens[i+1].split()
                order_by = [w.replace(" ", "") for w in order_by]
                self.order_by = (order_by[0],
                                 True if len(order_by) > 1 and order_by[1] == "ASC" else False)

    def match(self, record):
        for w in self.where:
            if not w[1](record[w[0]]): return False
        return True


In [36]:
from collections import defaultdict

class SQLStore:
    def __init__(self):
        self.rows = []
        self.indices = defaultdict(lambda: defaultdict(set))

    def insert(self, record):
        idx = len(self.rows)
        self.rows.append(record)
        for k, v in record.items():
            self.indices[k][v].add(idx)



In [37]:
class SQLEngine:
    def __init__(self, db):
        self.db = db

    def search(self, query):
        records = None
        for w in query.where:
            for v, r in self.db.indices[w[0]].items():
                if w[1](v):
                    if records == None:
                        records = set(r)
                    else:
                        records.intersection(r)
        results = [self.db.rows[r] for r in records]
        results.sort(key=lambda x: x[query.order_by[0]], reverse=query.order_by[1])
        results = [query.select(r) for r in results]

        return results

In [38]:
sql = SQLParser("SELECT name, age WHERE age < 11 ORDER BY name")
print(sql.select, sql.where, sql.order_by)

db = SQLStore()
db.insert({"name": "A", "age": 10, "zip": 1002})
db.insert({"name": "B", "age": 11, "zip": 1003})
db.insert({"name": "C", "age": 10, "zip": 1003})
print(db.rows, db.indices)

for i in range(3): print(sql.match(db.rows[i]))

e = SQLEngine(db)
e.search(sql)


operator.itemgetter('name', 'age') [('age', <function SQLParser._compile_where.<locals>.<lambda> at 0x794507a1fce0>)] ('name', False)
[{'name': 'A', 'age': 10, 'zip': 1002}, {'name': 'B', 'age': 11, 'zip': 1003}, {'name': 'C', 'age': 10, 'zip': 1003}] defaultdict(<function SQLStore.__init__.<locals>.<lambda> at 0x794507a1d580>, {'name': defaultdict(<class 'set'>, {'A': {0}, 'B': {1}, 'C': {2}}), 'age': defaultdict(<class 'set'>, {10: {0, 2}, 11: {1}}), 'zip': defaultdict(<class 'set'>, {1002: {0}, 1003: {1, 2}})})
True
False
True


[('A', 10), ('C', 10)]